In [45]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.ensemble import RandomForestClassifier, VotingClassifier, BaggingClassifier
from sklearn.neighbors import KNeighborsClassifier, RadiusNeighborsClassifier

from catboost import CatBoostRegressor
from lightgbm import LGBMRegressor
# import xgboost as xgb


import sklearn
sklearn.set_config(transform_output="pandas")

from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split, KFold
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score, mean_squared_log_error
from sklearn.preprocessing import OneHotEncoder, StandardScaler, RobustScaler, MinMaxScaler, OrdinalEncoder, TargetEncoder, FunctionTransformer
from sklearn.ensemble import StackingRegressor, VotingRegressor
from sklearn.linear_model import HuberRegressor
from sklearn.linear_model import Ridge

In [13]:
df = pd.read_csv('../train.csv')
df=df.drop_duplicates()
df = df.drop('Id', axis=1, errors='ignore') #убран из клинера
condition = (df['GrLivArea'] > 4000) & (df['SalePrice'] < 160001)
df=df[~condition]

Q1_price = df['SalePrice'].quantile(0.25) #очистка выбросов по цене
Q3_price = df['SalePrice'].quantile(0.75)
IQR_price = Q3_price - Q1_price
lower_bound_price = Q1_price - 1.5 * IQR_price
upper_bound_price = Q3_price + 1.5 * IQR_price
    
condition_price = (df['SalePrice'] < lower_bound_price) | (df['SalePrice'] > upper_bound_price)
df = df[~condition_price]

Q1_lot = df['LotArea'].quantile(0.25) #очистка выбросов по площади
Q3_lot = df['LotArea'].quantile(0.75)
IQR_lot = Q3_lot - Q1_lot
upper_bound_lot = Q3_lot + 3 * IQR_lot

condition_lot = df['LotArea'] > upper_bound_lot
df = df[~condition_lot]


cond_frontage = df['LotFrontage'] > 250 #LotFrontage - по рекомендации > 250 
df = df[~cond_frontage]

df['TotalPorchSF'] = (df['OpenPorchSF'] + df['EnclosedPorch'] + #Веранды
                          df['3SsnPorch'] + df['ScreenPorch'])
cond_porch = df['TotalPorchSF'] > 700
df = df[~cond_porch]
df = df.drop('TotalPorchSF', axis=1)
    

In [320]:
# quality_mapping = { вот это точно не работает результаты с этим энкодингом одинаковые, что и с встроенным
#     'Ex': 5,
#     'Gd': 4, 
#     'TA': 3,
#     'Fa': 2,
#     'Po': 1
# }
# quality_columns = ['ExterQual', 'ExterCond', 'BsmtQual', 'BsmtCond', 
#                    'HeatingQC', 'KitchenQual', 'FireplaceQu', 
#                    'GarageQual', 'GarageCond', 'PoolQC']
# for col in quality_columns:
#     if col in df.columns:
#         df[col] = df[col].map(quality_mapping)
#         df[col] = df[col].fillna(0)

# df['Total_qu'] = df['ExterQual'] + df['ExterCond'] + df['KitchenQual'] + df['HeatingQC'] + df['BsmtQual'] + df['BsmtCond'] + df['FireplaceQu'] + df['GarageQual'] + df['GarageCond'] + df['PoolQC']
# df['Quality_Area'] = df['OverallQual'] * df['GrLivArea']

In [14]:
X, y = df.drop('SalePrice', axis=1), df['SalePrice']

y_log = np.log1p(y)

X_train, X_valid, y_train_log, y_valid_log = train_test_split(X, y_log, test_size=0.2, random_state=42)

y_train_original = np.expm1(y_train_log)
y_valid_original = np.expm1(y_valid_log)

In [15]:
cat_features = ['MSZoning', 'Street', 'Alley', 'LotShape', 'LandContour', 'Utilities',
                'LotConfig', 'LandSlope', 'Neighborhood', 'Condition1', 'Condition2',
                'BldgType', 'HouseStyle', 'RoofStyle', 'RoofMatl', 'Exterior1st',
                'Exterior2nd', 'MasVnrType', 'Foundation', 'BsmtExposure', 'BsmtFinType1', 'BsmtFinType2',
                'Heating', 'CentralAir', 'Electrical', 'Functional', 'GarageType', 'GarageFinish',
                'PavedDrive', 'Fence', 'MiscFeature','SaleType', 'SaleCondition', 'PoolQC', 'ExterQual','ExterCond',
                'BsmtQual', 'BsmtCond','HeatingQC','KitchenQual','FireplaceQu', 'GarageQual', 'GarageCond']
num_features = ['MSSubClass', 'LotFrontage', 'LotArea', 'OverallQual',
                'OverallCond', 'YearBuilt', 'YearRemodAdd', 'MasVnrArea', 'BsmtFinSF1',
                'BsmtFinSF2', 'BsmtUnfSF', 'TotalBsmtSF', '1stFlrSF', '2ndFlrSF',
                'LowQualFinSF', 'GrLivArea', 'BsmtFullBath', 'BsmtHalfBath', 'FullBath',
                'HalfBath', 'BedroomAbvGr', 'KitchenAbvGr', 'TotRmsAbvGrd',
                'Fireplaces', 'GarageYrBlt', 'GarageCars', 'GarageArea', 'WoodDeckSF',
                'OpenPorchSF', 'EnclosedPorch', 'ScreenPorch', 'PoolArea', 'MoSold', 'YrSold', '3SsnPorch', 'MiscVal' 
                ] #начиная с PoolQC

In [52]:
def clean_data(X):
    X=X.copy()
    X['MasVnrType'] = X['MasVnrType'].fillna('None')
    X['FireplaceQu'] = X['FireplaceQu'].fillna('None')
    X['GarageQual'] = X['GarageQual'].fillna('None')
    X['GarageFinish'] = X['GarageFinish'].fillna('None')
    X['GarageType'] = X['GarageType'].fillna('None')
    X['GarageCond'] = X['GarageCond'].fillna('None')

    X['Alley'] = X['Alley'].fillna('None') #добавлена обработка
    X['PoolQC'] = X['PoolQC'].fillna('None') #добавлена обработка
    X['GarageType'] = X['GarageType'].fillna('None') #добавлена обработка

    X['GarageArea'] = X['GarageArea'].fillna(0) #добавлена обработка 
    X['MasVnrArea'] = X['MasVnrArea'].fillna(0) #добавлена обработка 

    X['GarageYrBlt'] = X['GarageYrBlt'].fillna(X['YearBuilt'])
    X['GarageAge'] = X['YrSold'] - X['GarageYrBlt']
    X['RemAge'] = X['YrSold'] - X['YearRemodAdd']
    X['HouseAge'] = X['YrSold'] - X['YearBuilt']
    X['IsNew'] = (X['HouseAge'] <= 5).astype(int)
    X['IsOld'] = (X['HouseAge'] >= 70).astype(int)
    X['IsHistoric'] = (X['HouseAge'] >= 100).astype(int)
    X['TotalBath'] = (X['FullBath'] + (0.5 * X['HalfBath']) +  #добавлено
                      X['BsmtFullBath'] + (0.5 * X['BsmtHalfBath']))
    X['HasPool'] = X['PoolArea'].apply(lambda x: 1 if x > 0 else 0)
    X['TotalSF'] = (X['GrLivArea'] + 
                    X['TotalBsmtSF'].fillna(0) + 
                    X['GarageArea'].fillna(0) +
                    X['WoodDeckSF'] + 
                    X['OpenPorchSF'] + 
                    X['EnclosedPorch'] + 
                    X['3SsnPorch'] + 
                    X['ScreenPorch'])
    X['TotalBathrooms'] = X['FullBath'] + 0.5*X['HalfBath'] + X['BsmtFullBath'] + 0.5*X['BsmtHalfBath']
    X['TotalRooms'] = X['TotRmsAbvGrd'] + X['BedroomAbvGr']
    X['AreaPerRoom'] = X['GrLivArea'] / (X['TotRmsAbvGrd'] + 1)
    X['QualityScore'] = X['OverallQual'] * X['OverallCond']
    X['IsRenovated'] = (X['YearRemodAdd'] != X['YearBuilt']).astype(int)
    X['YearsSinceRenovation'] = X['YrSold'] - X['YearRemodAdd']
    
    return X

def imputer_groupby_Neighborhood(X):
    X = X.copy()
    X['LotFrontage'] = X.groupby('Neighborhood')['LotFrontage'].transform(lambda x: x.fillna(x.median()))
    return X

cleaner = FunctionTransformer(clean_data)
group_imputer = FunctionTransformer(imputer_groupby_Neighborhood)

drop_features = ['MiscFeature', 'Utilities', 'PoolQC', 'Alley', 'Fence','Alley', 
                 'LowQualFinSF', 'MiscVal', 'HalfBath', 'BsmtHalfBath', 'Street', 'LandSlope'] 

# FireplaceQu_order = ['Ex', 'Gd', 'TA', 'Fa', 'Po']
# NEW обновили список колонок с учетом новообразованных
new_num_features = num_features + ['GarageAge', 'RemAge', 'HouseAge', 'TotalSF', 'TotalBath', 
                                   'IsNew', 'IsOld', 'IsHistoric', 'HasPool']

my_imputer = ColumnTransformer(
    transformers = [
        ('drop_features', 'drop', drop_features),
        ('num', Pipeline([
            ('imputer', SimpleImputer(strategy='median')),
            ('scaler', RobustScaler())
        ]), new_num_features),
        ('cat', Pipeline([
            ('imputer', SimpleImputer(strategy='most_frequent')),
            ('encoder', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
        ]), cat_features)
    ],
    verbose_feature_names_out = False,
    remainder = 'passthrough'
)


preprocessor = Pipeline(
    [
        ('cleaner', cleaner),
        ('group_imputer', group_imputer),
        ('imputer', my_imputer)
    ]
)

In [53]:
preprocessor.fit(X_train)
preprocessed_X_train = preprocessor.transform(X_train)
preprocessed_X_valid = preprocessor.transform(X_valid)

In [7]:
cb = CatBoostRegressor(
    cat_features=cat_features,
    iterations=2000,
    learning_rate=0.03,
    depth=5,
    l2_leaf_reg=12,
    bagging_temperature=1.0,
    random_strength=0.7507556902584652,
    subsample=0.8352580408542853,
    colsample_bylevel=0.9043302098880885,
    border_count=64,
    min_data_in_leaf = 40,
    eval_metric='RMSE',
    early_stopping_rounds=100,
    verbose=100,
    random_seed=42
)

cb.fit(
    preprocessed_X_train,
    y_train_log,
    eval_set=(preprocessed_X_valid, y_valid_log),
    plot=True
)



MetricVisualizer(layout=Layout(align_self='stretch', height='500px'))

0:	learn: 0.3469978	test: 0.3584441	best: 0.3584441 (0)	total: 86.2ms	remaining: 2m 52s
100:	learn: 0.1352018	test: 0.1503807	best: 0.1503807 (100)	total: 2.33s	remaining: 43.7s
200:	learn: 0.1124170	test: 0.1300606	best: 0.1300606 (200)	total: 3.82s	remaining: 34.2s
300:	learn: 0.1033576	test: 0.1225716	best: 0.1225716 (300)	total: 5.08s	remaining: 28.7s
400:	learn: 0.0980087	test: 0.1183484	best: 0.1183484 (400)	total: 6.5s	remaining: 25.9s
500:	learn: 0.0927182	test: 0.1147193	best: 0.1147124 (499)	total: 8.19s	remaining: 24.5s
600:	learn: 0.0899847	test: 0.1135834	best: 0.1135834 (600)	total: 9.88s	remaining: 23s
700:	learn: 0.0876531	test: 0.1127011	best: 0.1126750 (694)	total: 11s	remaining: 20.4s
800:	learn: 0.0859601	test: 0.1122172	best: 0.1122172 (800)	total: 12.2s	remaining: 18.2s
900:	learn: 0.0843781	test: 0.1118123	best: 0.1118123 (900)	total: 13.2s	remaining: 16.1s
1000:	learn: 0.0826915	test: 0.1114820	best: 0.1114769 (998)	total: 15s	remaining: 15s
1100:	learn: 0.08056

CatBoostRegressor(bagging_temperature=1.0, border_count=64, cat_features=['MSZoning', 'Street', 'Alley', 'LotShape', 'LandContour', 'Utilities', 'LotConfig', 'LandSlope', 'Neighborhood', 'Condition1', 'Condition2', 'BldgType', 'HouseStyle', 'RoofStyle', 'RoofMatl', 'Exterior1st', 'Exterior2nd', 'MasVnrType', 'Foundation', 'BsmtExposure', 'BsmtFinType1', 'BsmtFinType2', 'Heating', 'CentralAir', 'Electrical', 'Functional', 'GarageType', 'GarageFinish', 'PavedDrive', 'Fence', 'MiscFeature', 'SaleType', 'SaleCondition', 'PoolQC', 'ExterQual', 'ExterCond', 'BsmtQual', 'BsmtCond', 'HeatingQC', 'KitchenQual', 'FireplaceQu', 'GarageQual', 'GarageCond'], colsample_bylevel=0.9043302098880885, depth=5, early_stopping_rounds=100, eval_metric='RMSE', iterations=2000, l2_leaf_reg=12, learning_rate=0.03, loss_function='RMSE', min_data_in_leaf=40, random_seed=42, random_strength=0.7507556902584652, subsample=0.8352580408542853, verbose=100)

In [8]:
y_pred_train_log = cb.predict(preprocessed_X_train)
y_pred_valid_log = cb.predict(preprocessed_X_valid)

# Возвращаем к исходному масштабу
y_pred_train = np.expm1(y_pred_train_log)
y_pred_valid = np.expm1(y_pred_valid_log)

In [328]:
final_pipeline = Pipeline([('preprocessor', preprocessor), ('CatBoost', cb)])
final_pipeline.fit(X_train, y_train_log)

0:	learn: 0.3469978	total: 11.1ms	remaining: 22.1s
100:	learn: 0.1352018	total: 709ms	remaining: 13.3s
200:	learn: 0.1124170	total: 1.71s	remaining: 15.3s
300:	learn: 0.1033576	total: 2.6s	remaining: 14.7s
400:	learn: 0.0980087	total: 3.3s	remaining: 13.2s
500:	learn: 0.0927182	total: 4.02s	remaining: 12s
600:	learn: 0.0899847	total: 4.72s	remaining: 11s
700:	learn: 0.0876531	total: 5.44s	remaining: 10.1s
800:	learn: 0.0859601	total: 6.26s	remaining: 9.37s
900:	learn: 0.0843781	total: 6.99s	remaining: 8.53s
1000:	learn: 0.0826915	total: 7.72s	remaining: 7.7s
1100:	learn: 0.0805664	total: 8.77s	remaining: 7.16s
1200:	learn: 0.0781176	total: 9.62s	remaining: 6.4s
1300:	learn: 0.0760538	total: 10.4s	remaining: 5.57s
1400:	learn: 0.0740790	total: 11.1s	remaining: 4.75s
1500:	learn: 0.0722979	total: 11.8s	remaining: 3.94s
1600:	learn: 0.0702932	total: 12.6s	remaining: 3.14s
1700:	learn: 0.0687325	total: 13.4s	remaining: 2.35s
1800:	learn: 0.0674260	total: 14.1s	remaining: 1.56s
1900:	learn:

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators ` for more details.","[('preprocessor', ...), ('CatBoost', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing `.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators ` for more details.","[('cleaner', ...), ('group_imputer', ...), ...]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing `.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"func func: callable, default=NoneThe callable to use for the transformation. This will be passedthe same arguments as transform, with args and kwargs forwarded.If func is None, then func will be the identity function.",<function cle...x78d6f0cd0360>
,"inverse_func inverse_func: callable, default=NoneThe callable to use for the 

In [56]:
train_rmse = np.sqrt(mean_squared_error(y_train_original, y_pred_train))
train_mae = mean_absolute_error(y_train_original, y_pred_train)
train_r2 = r2_score(y_train_original, y_pred_train)

# Метрики для validation
valid_rmse = np.sqrt(mean_squared_error(y_valid_original, y_pred_valid))
valid_mae = mean_absolute_error(y_valid_original, y_pred_valid)
valid_r2 = r2_score(y_valid_original, y_pred_valid)

print("Train Metrics:")
print(f"RMSE: {train_rmse:.4f}")
print(f"MAE: {train_mae:.4f}")
print(f"R²: {train_r2:.4f}")
print("\nValidation Metrics:")
print(f"RMSE: {valid_rmse:.4f}")
print(f"MAE: {valid_mae:.4f}")
print(f"R²: {valid_r2:.4f}")

Train Metrics:
RMSE: 8327.4137
MAE: 6176.5349
R²: 0.9800

Validation Metrics:
RMSE: 15538.9467
MAE: 10789.1412
R²: 0.9312


In [57]:
base_models = [
    ('catboost', CatBoostRegressor(iterations=1000, 
        learning_rate=0.03, # Уменьшили шаг
        depth=6, 
        l2_leaf_reg=5,      # Добавили регуляризацию
        verbose=0)),
    ('lightgbm', LGBMRegressor(n_estimators=1000, 
        learning_rate=0.03, 
        num_leaves=31, 
        reg_lambda=10,      # L2 регуляризация
        reg_alpha=1,        # L1 регуляризация
        subsample=0.8))
]

final_model = Ridge()

stacking_model = StackingRegressor(
    estimators=base_models,
    final_estimator=final_model,
    cv=5 # Кросс-валидация для обучения мета-модели
)
stacking_model.fit(preprocessed_X_train, y_train_log)

[LightGBM] [Warning] Found whitespace in feature_names, replace with underlines
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.001103 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 3996
[LightGBM] [Info] Number of data points in the train set: 1092, number of used features: 190
[LightGBM] [Info] Start training from score 11.983612
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive

,"estimators estimators: list of (str, estimator)Base estimators which will be stacked together. Each element of thelist is defined as a tuple of string (i.e. name) and an estimatorinstance. An estimator can be set to 'drop' using `set_params`.","[('catboost', ...), ('lightgbm', ...)]"
,"final_estimator final_estimator: estimator, default=NoneA regressor which will be used to combine the base estimators.The default regressor is a :class:`~sklearn.linear_model.RidgeCV`.",Ridge()
,"cv cv: int, cross-validation generator, iterable, or ""prefit"", default=NoneDetermines the cross-validation splitting strategy used in`cross_val_predict` to train `final_estimator`. Possible inputs forcv are:* None, to use the default 5-fold cross validation,* integer, to specify the number of folds in a (Stratified) KFold,* An object to be used as a cross-validation generator,* An iterable yielding train, test splits,* `""prefit""`, to assume the `estimators` are prefit. In this case, the estimators will not be refitted.For integer/None inputs, if the estimator is a classifier and y iseither binary or multiclass,:class:`~sklearn.model_selection.StratifiedKFold` is used.In all other cases, :class:`~sklearn.model_selection.KFold` is used.These splitters are instantiated with `shuffle=False` so the splitswill be the same across calls.Refer :ref:`User Guide ` for the variouscross-validation strategies that can be used here.If ""prefit"" is passed, it is assumed that all `estimators` havebeen fitted already. The `final_estimator_` is trained on the `estimators`predictions on the full training set and are **not** cross validatedpredictions. Please note that if the models have been trained on the samedata to train the stacking model, there is a very high risk of overfitting... versionadded:: 1.1 The 'prefit' option was added in 1.1.. note:: A larger number of split will provide no benefits if the number of training samples is large enough. Indeed, the training time will increase. ``cv`` is not used for model evaluation but for prediction.",5
,"n_jobs n_jobs: int, default=NoneThe number of jobs to run in parallel for `fit` of all `estimators`.`None` means 1 unless in a `joblib.parallel_backend` context. -1 meansusing all processors. See :term:`Glossary ` for more details.",None
,"passthrough passthrough: bool, default=FalseWhen False, only the predictions of estimators will be used astraining data for `final_estimator`. When True, the`final_estimator` is trained on the predictions as well as theoriginal training data.",False
,"verbose verbose: int, default=0Verbosity level.",0
,boosting_type,'gbdt'
,num_leaves,31
,max_depth,-1
,learning_rate,0.03
,n_estimators,1000


In [58]:
y_pred_train_log = stacking_model.predict(preprocessed_X_train)
y_pred_valid_log = stacking_model.predict(preprocessed_X_valid)

# Возвращаем к исходному масштабу
y_pred_train = np.expm1(y_pred_train_log)
y_pred_valid = np.expm1(y_pred_valid_log)

/home/ali_b_indahouse/ds_bootcamp/hw/proj_ml/House-Prices/env/lib/python3.12/site-packages/sklearn/utils/validation.py:2684: UserWarning: X has feature names, but Ridge was fitted without feature names
  warnings.warn(
/home/ali_b_indahouse/ds_bootcamp/hw/proj_ml/House-Prices/env/lib/python3.12/site-packages/sklearn/utils/validation.py:2684: UserWarning: X has feature names, but Ridge was fitted without feature names
  warnings.warn(


In [59]:
# 1. Снижаем агрессивность базовых моделей
cat = CatBoostRegressor(iterations=1000, learning_rate=0.03, depth=5, l2_leaf_reg=30, random_strength=2, bagging_temperature=1, verbose=0)
lgbm = LGBMRegressor(n_estimators=800, learning_rate=0.05, reg_alpha=5, reg_lambda=5)
# Линейная модель для баланса
HuberRegr = HuberRegressor(alpha=1.0) 

voting_model = VotingRegressor(
    estimators=[
        ('cat', cat),
        ('lgbm', lgbm),
        ('hr', HuberRegr)
    ],
    weights=[0.4, 0.4, 0.2] # 80% бустингам, 20% линейной регрессии
)

voting_model.fit(preprocessed_X_train, y_train_log)

[LightGBM] [Warning] Found whitespace in feature_names, replace with underlines
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.001466 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 3996
[LightGBM] [Info] Number of data points in the train set: 1092, number of used features: 190
[LightGBM] [Info] Start training from score 11.983612
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive

/home/ali_b_indahouse/ds_bootcamp/hw/proj_ml/House-Prices/env/lib/python3.12/site-packages/sklearn/linear_model/_huber.py:348: ConvergenceWarning: lbfgs failed to converge after 100 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=100).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
  self.n_iter_ = _check_optimize_result("lbfgs", opt_res, self.max_iter)


,"estimators estimators: list of (str, estimator) tuplesInvoking the ``fit`` method on the ``VotingRegressor`` will fit clonesof those original estimators that will be stored in the class attribute``self.estimators_``. An estimator can be set to ``'drop'`` using:meth:`set_params`... versionchanged:: 0.21 ``'drop'`` is accepted. Using None was deprecated in 0.22 and support was removed in 0.24.","[('cat', ...), ('lgbm', ...), ...]"
,"weights weights: array-like of shape (n_regressors,), default=NoneSequence of weights (`float` or `int`) to weight the occurrences ofpredicted values before averaging. Uses uniform weights if `None`.","[0.4, 0.4, ...]"
,"n_jobs n_jobs: int, default=NoneThe number of jobs to run in parallel for ``fit``.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary `for more details.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting will be printed as itis completed... versionadded:: 0.23",False
,boosting_type,'gbdt'
,num_leaves,31
,max_depth,-1
,learning_rate,0.05
,n_estimators,800
,subsample_for_bin,200000
,objective,None


In [60]:
y_pred_train_log = voting_model.predict(preprocessed_X_train)
y_pred_valid_log = voting_model.predict(preprocessed_X_valid)

# Возвращаем к исходному масштабу
y_pred_train = np.expm1(y_pred_train_log)
y_pred_valid = np.expm1(y_pred_valid_log)

In [61]:
train_rmse = np.sqrt(mean_squared_error(y_train_original, y_pred_train))
train_mae = mean_absolute_error(y_train_original, y_pred_train)
train_r2 = r2_score(y_train_original, y_pred_train)

# Метрики для validation
valid_rmse = np.sqrt(mean_squared_error(y_valid_original, y_pred_valid))
valid_mae = mean_absolute_error(y_valid_original, y_pred_valid)
valid_r2 = r2_score(y_valid_original, y_pred_valid)

print("Train Metrics:")
print(f"RMSE: {train_rmse:.4f}")
print(f"MAE: {train_mae:.4f}")
print(f"R²: {train_r2:.4f}")
print("\nValidation Metrics:")
print(f"RMSE: {valid_rmse:.4f}")
print(f"MAE: {valid_mae:.4f}")
print(f"R²: {valid_r2:.4f}")

Train Metrics:
RMSE: 28756.0205
MAE: 20881.6981
R²: 0.7615

Validation Metrics:
RMSE: 35672.7976
MAE: 23482.2364
R²: 0.6374


In [ ]:
test = pd.read_csv('../test.csv')

In [ ]:
result = np.expm1(final_pipeline.predict(test))
df_result = pd.DataFrame(test['Id'])
df_result['SalePrice'] = result

In [ ]:
df_result.to_csv('submission.csv', index=False)

In [1529]:
# lgbm = lgb.LGBMRegressor(
#     objective='regression',           # бинарная классификация
#     boosting_type='gbdt',
#     max_depth = 3,                # тип бустинга
#     num_leaves=112,                # количество листьев
#     learning_rate=0.01692071083311199,            # скорость обучения
#     min_child_weight = 0.012448732459530546,
#     min_split_gain = 0.004071232392911059,
#     n_estimators=600,             # количество деревьев
#     random_state=42,
#     subsample = 0.6742497703954182, 
#     subsample_freq = 9,
#     colsample_bytree = 0.8504334860826162,
#     colsample_bynode = 0.8912182644144621, 
#     reg_alpha = 0.2680177236276531, 
#     reg_lambda = 0.013509588482037907, 
#     min_child_samples = 6,
#     verbose=-1                    # подавляем вывод
# )


# lgbm.fit(
#     preprocessed_X_train,
#     y_train_log,
#     eval_set=[(preprocessed_X_valid, y_valid_log)],
#     eval_metric='rmse',
#     callbacks=[lgb.early_stopping(stopping_rounds=50),   # ранняя остановка
#                lgb.log_evaluation(period=10)]            # логирование каждые 10 итераций
# )

In [1530]:
# y_pred_train_log = lgbm.predict(preprocessed_X_train)
# y_pred_valid_log = lgbm.predict(preprocessed_X_valid)

# # Возвращаем к исходному масштабу
# y_pred_train = np.expm1(y_pred_train_log)
# y_pred_valid = np.expm1(y_pred_valid_log)

# train_rmse = np.sqrt(mean_squared_error(y_train_original, y_pred_train))
# train_mae = mean_absolute_error(y_train_original, y_pred_train)
# train_r2 = r2_score(y_train_original, y_pred_train)

# # Метрики для validation
# valid_rmse = np.sqrt(mean_squared_error(y_valid_original, y_pred_valid))
# valid_mae = mean_absolute_error(y_valid_original, y_pred_valid)
# valid_r2 = r2_score(y_valid_original, y_pred_valid)

# print("Train Metrics:")
# print(f"RMSE: {train_rmse:.4f}")
# print(f"MAE: {train_mae:.4f}")
# print(f"R²: {train_r2:.4f}")
# print("\nValidation Metrics:")
# print(f"RMSE: {valid_rmse:.4f}")
# print(f"MAE: {valid_mae:.4f}")
# print(f"R²: {valid_r2:.4f}")

In [1531]:
import optuna
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from catboost import CatBoostRegressor
import lightgbm as lgb
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import math

# Предполагается, что у вас уже есть:
# X_train, X_valid, y_train, y_valid (в исходном масштабе)
# Или X_train_log, y_train_log, X_valid_log, y_valid_log

# Если вы обучаетесь на логарифмах, определите функцию inverse transform
def inv_transform(y_pred_log, y_true=None):
    """Преобразует предсказания из логарифмического масштаба обратно"""
    y_pred = np.expm1(y_pred_log)  # если использовали log1p, иначе np.exp
    return y_pred

In [1532]:
# def objective_catboost(trial, X_train, y_train, X_valid, y_valid):
#     """
#     Целевая функция для оптимизации CatBoost
#     """
#     params = {
#         'iterations': trial.suggest_int('iterations', 300, 1000, step=50),
#         'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.3, log=True),
#         'depth': trial.suggest_int('depth', 3, 8),
#         'l2_leaf_reg': trial.suggest_float('l2_leaf_reg', 1, 15, log=True),
#         'bagging_temperature': trial.suggest_float('bagging_temperature', 0.5, 3.0),
#         'random_strength': trial.suggest_float('random_strength', 0.5, 3.0),
#         'subsample': trial.suggest_float('subsample', 0.6, 0.95),
#         'colsample_bylevel': trial.suggest_float('colsample_bylevel', 0.7, 1.0),
#         'border_count': trial.suggest_int('border_count', 32, 255, step=32),
#         'min_data_in_leaf': trial.suggest_int('min_data_in_leaf', 10, 100, step=10),
#         'eval_metric': 'RMSE',
#         'early_stopping_rounds': 50,
#         'verbose': 0,
#         'random_seed': 42
#     }
    
#     # Создаем и обучаем модель
#     model = CatBoostRegressor(**params)
#     model.fit(
#         X_train, y_train,
#         eval_set=(X_valid, y_valid),
#         verbose=False
#     )
    
#     # Получаем предсказания на валидации
#     y_pred = model.predict(X_valid)
    
#     # Вычисляем метрику (RMSE)
#     rmse = np.sqrt(mean_squared_error(y_valid, y_pred))
    
#     return rmse

# def optimize_catboost(X_train, y_train, X_valid, y_valid, n_trials=50):
#     """
#     Запуск оптимизации CatBoost
#     """
#     study = optuna.create_study(
#         direction='minimize',
#         sampler=optuna.samplers.TPESampler(seed=42),
#         pruner=optuna.pruners.MedianPruner()
#     )
    
#     study.optimize(
#         lambda trial: objective_catboost(trial, X_train, y_train, X_valid, y_valid),
#         n_trials=n_trials,
#         show_progress_bar=True
#     )
    
#     print("Best trial:")
#     print(f"  RMSE: {study.best_value:.4f}")
#     print("  Params:")
#     for key, value in study.best_params.items():
#         print(f"    {key}: {value}")
    
#     return study

# study_catboost = optimize_catboost(
#     preprocessed_X_train, y_train_log,
#     preprocessed_X_valid, y_valid_log,
#     n_trials=200
# )

In [1533]:
# import optuna
# import numpy as np
# from sklearn.metrics import mean_squared_error
# from catboost import CatBoostRegressor

# def preprocess_categorical_features(X_train, X_valid):
#     """
#     Заполняет NaN в категориальных колонках
#     """
#     # Создаем копии, чтобы не изменять исходные данные
#     X_train_clean = X_train.copy()
#     X_valid_clean = X_valid.copy()
    
#     # Находим категориальные колонки
#     categorical_cols = X_train_clean.select_dtypes(include=['object', 'category']).columns.tolist()
    
#     # Заполняем NaN в категориальных колонках
#     for col in categorical_cols:
#         # Заменяем NaN на специальную строку
#         X_train_clean[col] = X_train_clean[col].fillna('__MISSING__')
#         X_valid_clean[col] = X_valid_clean[col].fillna('__MISSING__')
        
#         # Убеждаемся, что тип данных - category
#         X_train_clean[col] = X_train_clean[col].astype('category')
#         X_valid_clean[col] = X_valid_clean[col].astype('category')
    
#     return X_train_clean, X_valid_clean, categorical_cols

# # Применяем предобработку - создаем очищенные данные
# X_train_clean, X_valid_clean, categorical_cols = preprocess_categorical_features(X_train, X_valid)

# print(f"Категориальные колонки после обработки: {len(categorical_cols)}")
# print(f"Пример заполненных значений:")
# for col in categorical_cols[:3]:
#     print(f"  {col}: {X_train_clean[col].value_counts().head(3)}")
#     print(f"    Тип: {X_train_clean[col].dtype}")

# # Исправленная функция objective - использует очищенные данные
# def objective_with_stability_clean(trial, X_train_clean, y_train, X_valid_clean, y_valid, categorical_cols):
#     """
#     Оптимизация с учетом разрыва между train и valid (использует предобработанные данные)
#     """
#     params = {
#         'depth': 4,
#         'border_count': 64,
#         'iterations': trial.suggest_int('iterations', 800, 1200, step=50),
#         'learning_rate': trial.suggest_float('learning_rate', 0.05, 0.08, log=True),
#         'l2_leaf_reg': trial.suggest_float('l2_leaf_reg', 8, 11, log=True),
#         'bagging_temperature': trial.suggest_float('bagging_temperature', 0.7, 1.0),
#         'random_strength': trial.suggest_float('random_strength', 1.5, 2.0),
#         'subsample': trial.suggest_float('subsample', 0.62, 0.72),
#         'colsample_bylevel': trial.suggest_float('colsample_bylevel', 0.68, 0.75),
#         'min_data_in_leaf': trial.suggest_int('min_data_in_leaf', 70, 95, step=5),
#         'eval_metric': 'RMSE',
#         'early_stopping_rounds': 50,
#         'verbose': 0,
#         'random_seed': 42,
#         'cat_features': categorical_cols
#     }
    
#     try:
#         model = CatBoostRegressor(**params)
#         model.fit(
#             X_train_clean, y_train,
#             eval_set=(X_valid_clean, y_valid),
#             verbose=False
#         )
        
#         # Предсказания
#         y_train_pred = model.predict(X_train_clean)
#         y_valid_pred = model.predict(X_valid_clean)
        
#         # Метрики
#         train_rmse = np.sqrt(mean_squared_error(y_train, y_train_pred))
#         valid_rmse = np.sqrt(mean_squared_error(y_valid, y_valid_pred))
        
#         # Штраф за переобучение
#         overfit_ratio = train_rmse / valid_rmse
#         if overfit_ratio < 0.85:
#             penalty = (0.85 - overfit_ratio) * 2
#         else:
#             penalty = 0
        
#         # Итоговая метрика
#         score = valid_rmse * (1 + penalty)
        
#         return score
    
#     except Exception as e:
#         print(f"Trial failed: {e}")
#         return float('inf')

# # Запуск оптимизации с очищенными данными
# print("\n" + "="*60)
# print("Запуск оптимизации с предобработанными категориальными признаками")
# print("="*60)

# study_stability = optuna.create_study(direction='minimize')
# study_stability.optimize(
#     lambda trial: objective_with_stability_clean(
#         trial, X_train_clean, y_train, X_valid_clean, y_valid, categorical_cols
#     ),
#     n_trials=200  # Начните с 50, затем увеличьте до 200 если нужно
# )

# # Результаты
# print("\n" + "="*60)
# print("ОПТИМИЗАЦИЯ ЗАВЕРШЕНА")
# print("="*60)
# print(f"Лучшее значение (penalized RMSE): {study_stability.best_value:.6f}")
# print(f"\nЛучшие параметры:")
# for key, value in study_stability.best_params.items():
#     print(f"  {key:20s}: {value}")

# # Сравнение с первым этапом
# first_stage_params = {
#     'iterations': 1000,
#     'learning_rate': 0.06591629019374852,
#     'l2_leaf_reg': 9.516009955450892,
#     'bagging_temperature': 0.8088519508440091,
#     'random_strength': 1.7382976428044792,
#     'subsample': 0.6691034225858007,
#     'colsample_bylevel': 0.7064617749495126,
#     'min_data_in_leaf': 80
# }

# print("\n" + "="*60)
# print("СРАВНЕНИЕ С ПЕРВЫМ ЭТАПОМ ОПТИМИЗАЦИИ")
# print("="*60)
# print(f"\n{'Параметр':<25} {'Первый этап':>15} {'Второй этап':>15} {'Изменение':>15}")
# print("-"*70)

# for param in first_stage_params.keys():
#     first_val = first_stage_params[param]
#     second_val = study_stability.best_params.get(param)
#     if second_val:
#         change = ((second_val - first_val) / first_val) * 100
#         print(f"{param:<25} {first_val:>15.6f} {second_val:>15.6f} {change:>14.2f}%")

# # Опционально: визуализация
# try:
#     from optuna.visualization import plot_optimization_history, plot_param_importances
    
#     fig1 = plot_optimization_history(study_stability)
#     fig1.show()
    
#     fig2 = plot_param_importances(study_stability)
#     fig2.show()
# except:
#     print("Для визуализации установите plotly: pip install plotly")

In [1534]:
# def objective_lgbm(trial, X_train, y_train, X_valid, y_valid):
#     """
#     Целевая функция для оптимизации LightGBM
#     """
#     params = {
#         'objective': 'regression',
#         'boosting_type': trial.suggest_categorical('boosting_type', ['gbdt', 'dart', 'goss']),
#         'metric': 'rmse',
#         'n_estimators': trial.suggest_int('n_estimators', 300, 1500, step=50),
#         'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.3, log=True),
#         'num_leaves': trial.suggest_int('num_leaves', 31, 255),
#         'max_depth': trial.suggest_int('max_depth', 3, 12),
#         'min_child_samples': trial.suggest_int('min_child_samples', 10, 100, step=10),
#         'min_child_weight': trial.suggest_float('min_child_weight', 0.001, 0.1, log=True),
#         'min_split_gain': trial.suggest_float('min_split_gain', 0.0, 0.05),
#         'subsample': trial.suggest_float('subsample', 0.6, 1.0),
#         'subsample_freq': trial.suggest_int('subsample_freq', 1, 10),
#         'colsample_bytree': trial.suggest_float('colsample_bytree', 0.6, 1.0),
#         'colsample_bynode': trial.suggest_float('colsample_bynode', 0.6, 1.0),
#         'reg_alpha': trial.suggest_float('reg_alpha', 0.0, 1.0, log=True),
#         'reg_lambda': trial.suggest_float('reg_lambda', 0.0, 1.0, log=True),
#         'random_state': 42,
#         'verbose': -1
#     }
    
#     # Особые настройки для boosting_type='dart'
#     if params['boosting_type'] == 'dart':
#         params['drop_rate'] = trial.suggest_float('drop_rate', 0.05, 0.3)
#         params['max_drop'] = trial.suggest_int('max_drop', 10, 50)
#         params['skip_drop'] = trial.suggest_float('skip_drop', 0.3, 0.7)
    
#     # Ранняя остановка через callback
#     model = lgb.LGBMRegressor(**params)
    
#     model.fit(
#         X_train, y_train,
#         eval_set=[(X_valid, y_valid)],
#         eval_metric='rmse',
#         callbacks=[
#             lgb.early_stopping(stopping_rounds=50),
#             lgb.log_evaluation(0)  # отключаем вывод
#         ]
#     )
    
#     # Получаем предсказания
#     y_pred = model.predict(X_valid)
    
#     # Вычисляем RMSE
#     rmse = np.sqrt(mean_squared_error(y_valid, y_pred))
    
#     return rmse

# def optimize_lgbm(X_train, y_train, X_valid, y_valid, n_trials=50):
#     """
#     Запуск оптимизации LightGBM
#     """
#     study = optuna.create_study(
#         direction='minimize',
#         sampler=optuna.samplers.TPESampler(seed=42),
#         pruner=optuna.pruners.MedianPruner()
#     )
    
#     study.optimize(
#         lambda trial: objective_lgbm(trial, X_train, y_train, X_valid, y_valid),
#         n_trials=n_trials,
#         show_progress_bar=True
#     )
    
#     print("Best trial:")
#     print(f"  RMSE: {study.best_value:.4f}")
#     print("  Params:")
#     for key, value in study.best_params.items():
#         print(f"    {key}: {value}")
    
#     return study

# study_lgbm = optimize_lgbm(
#     preprocessed_X_train, y_train_log,
#     preprocessed_X_valid, y_valid_log,
#     n_trials=200
# )